# 04 Baseline by Otsu thresholding
Otsu is a segementation technique 
1. Intensities are normalised (z-score).
2. Otsu thresholding separates the bright blood from the rest, inside a box around the LA positions of the training set. With 2 classes, Otsu mostly separates air/lungs from tissue, so we also try 3 classes (air, tissue, blood) and keep the brightest one.
3. A morphological opening cuts the connections between the LA and the other bright structures (ventricle, right atrium, aorta).
4. Only the connected component closest to the mean LA centroid of the training set is kept.

The number of Otsu classes and the opening radius are chosen by 5-fold cross-validation on the development set. The test set is not used here.

In [1]:
import json
import sys

import numpy as np
import pandas as pd

# shared functions of the project (src/utils.py)
sys.path.insert(0, "../src")
from utils import RESULTS, SPLITS, evaluate, fold_cases, la_prior, load_case, normalize, otsu_segment

# values tried in cross-validation
N_CLASSES = [2, 3]  # number of Otsu classes
RADII = [0, 3, 5, 7, 9, 11]  # opening radius in voxels (0 = no opening)

# load the 15 development volumes once, with normalised intensities
data = {}
for c in SPLITS["dev"]:
    x, y, spacing = load_case(c)
    data[c] = (normalize(x), y, spacing)

## Cross-validation

In [2]:
rows = []
for k in range(len(SPLITS["folds"])):
    # 12 volumes to "train" (here: compute the LA position), 3 volumes to evaluate
    train, val = fold_cases(k)
    # the LA position prior only comes from the training volumes of the fold
    centroid, box = la_prior([data[c][1] for c in train])
    for c in val:
        x, y, spacing = data[c]
        # every combination of settings is tested on every validation volume
        for n in N_CLASSES:
            for r in RADII:
                pred = otsu_segment(x, centroid, box, n, r)
                rows.append({"fold": k, "case": c, "n_classes": n, "radius": r, **evaluate(pred, y, spacing)})

# one row per (volume, setting): saved, then averaged per setting
cv = pd.DataFrame(rows)
cv.to_csv(RESULTS / "cv_otsu.csv", index=False)
cv.groupby(["n_classes", "radius"])[["dice", "hd95_mm", "sensitivity", "precision", "volume_error_ml"]].mean().round(3)

dice  hd95_mm  sensitivity  precision  volume_error_ml
n_classes radius                                                         
2         0       0.185   66.054        0.996      0.103          872.530
          3       0.223   62.731        0.994      0.126          687.820
          5       0.248   59.188        0.971      0.143          579.380
          7       0.253   55.284        0.872      0.149          462.977
          9       0.315   53.431        0.873      0.193          354.531
          11      0.386   51.506        0.792      0.261          222.534
3         0       0.264   56.896        0.951      0.154          515.076
          3       0.280   52.901        0.860      0.169          385.470
          5       0.302   50.871        0.786      0.191          308.909
          7       0.347   49.711        0.677      0.241          185.368
          9       0.429   43.799        0.524      0.427           40.140
          11      0.386   41.602        0.350      0.565          -30.993

In [3]:
# chosen configuration = best mean Dice over the 15 validation volumes
n_best, r_best = cv.groupby(["n_classes", "radius"])["dice"].mean().idxmax()
print(f"chosen: {n_best} classes, radius {r_best}")

# results of the chosen setting for each volume
best = cv[(cv["n_classes"] == n_best) & (cv["radius"] == r_best)]
best.round(3)

chosen: 3 classes, radius 9


,fold,case,n_classes,radius,dice,hd95_mm,sensitivity,precision,volume_ml,ref_volume_ml,volume_error_ml
10,0,la_003,3,9,0.404,44.963,0.696,0.285,235.390,96.335,139.055
22,0,la_010,3,9,0.664,26.377,0.525,0.901,46.978,80.526,-33.548
34,0,la_023,3,9,0.589,18.713,0.432,0.925,43.213,92.484,-49.271
46,1,la_005,3,9,0.215,52.839,0.171,0.289,73.856,124.608,-50.752
58,1,la_017,3,9,0.387,54.623,0.500,0.316,127.926,80.871,47.055
70,1,la_022,3,9,0.493,56.084,0.768,0.363,151.888,71.820,80.068
82,1,la_026,3,9,0.225,53.149,0.185,0.288,73.856,115.305,-41.449
94,2,la_007,3,9,0.385,46.363,0.447,0.338,157.079,118.685,38.394
106,2,la_014,3,9,0.411,38.522,0.427,0.395,157.753,145.879,11.874
118,2,la_024,3,9,0.397,42.820,0.724,0.274,263.312,99.518,163.794


In [4]:
# save the chosen configuration (used by the final training and test notebooks)
config_file = RESULTS / "chosen_configs.json"
configs = json.loads(config_file.read_text()) if config_file.exists() else {}
configs["otsu"] = {"n_classes": int(n_best), "radius": int(r_best)}
config_file.write_text(json.dumps(configs, indent=2))

148